In [1]:
import pandas as pd

In [2]:
import kaleidocell

In [5]:
genes_df = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea/genes.csv")
print(genes_df.head())
print(genes_df.columns.tolist())

     gene   mp     score
0   SKAP1  MP1  0.012087
1    CD96  MP1  0.008915
2     FYN  MP1  0.008811
3  THEMIS  MP1  0.008810
4   CD247  MP1  0.007919
['gene', 'mp', 'score']


In [6]:
# Check distribution of scores
print(genes_df["score"].describe())

# Top 10 genes per MP
#top_genes_per_mp = genes_df.sort_values("score", ascending=False).groupby("mp").head(10)
top_genes_per_mp = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_3_top10_genes_per_mp.csv")

#top_genes_per_mp.to_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/top10_genes_per_mp.csv", index=False)

count    12442.000000
mean         0.000365
std          0.001566
min          0.000023
25%          0.000034
50%          0.000042
75%          0.000067
max          0.038601
Name: score, dtype: float64


In [7]:
gsea_df = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea/gsea_C7.csv")

In [8]:
results_mp = kaleidocell.load("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea/results_mp.kc")
metrics = results_mp["metrics"]
print(metrics)

      sampleCoverage  silhouette  meanSimilarity  nPrograms  nGenes
MP1         0.149573    0.279149           0.491        196      55
MP2         0.495726    0.357757           0.541        571      41
MP3         0.341880    0.335111           0.433        284       6
MP4         0.205128    0.146454           0.327        149      13
MP5         0.273504    0.098320           0.283        193       8
MP6         0.478632   -0.018113           0.224        512      60
MP7         0.081197    0.284584           0.427         92      28
MP8         0.329060    0.050496           0.267        304      30
MP9         0.294872    0.026419           0.266        227      26
MP10        0.230769    0.230069           0.429        135       2
MP11        0.303419    0.405800           0.519        200      94
MP12        0.264957    0.393355           0.490        228      10
MP13        0.299145    0.068925           0.246        158       2
MP14        0.487179    0.050864           0.244

In [9]:
# Check the best performing MPs based on silhouette score
print(results_mp["metrics"].sort_values("silhouette", ascending=False))

      sampleCoverage  silhouette  meanSimilarity  nPrograms  nGenes
MP11        0.303419    0.405800           0.519        200      94
MP12        0.264957    0.393355           0.490        228      10
MP2         0.495726    0.357757           0.541        571      41
MP24        0.038462    0.340832           0.361         58      43
MP3         0.341880    0.335111           0.433        284       6
MP7         0.081197    0.284584           0.427         92      28
MP1         0.149573    0.279149           0.491        196      55
MP10        0.230769    0.230069           0.429        135       2
MP20        0.047009    0.216024           0.337         53      36
MP22        0.132479    0.205681           0.298        174      10
MP4         0.205128    0.146454           0.327        149      13
MP19        0.102564    0.118333           0.227        100      17
MP5         0.273504    0.098320           0.283        193       8
MP30        0.260684    0.088221           0.255

In [10]:
# 1. Max gene score per MP
mp_max_score = genes_df.groupby("mp")["score"].max().sort_values(ascending=False)
print("Max gene score per MP:")
print(mp_max_score)

# 2. Number of significant pathways per MP
mp_pathway_counts = gsea_df.groupby("MP").size().sort_values(ascending=False)
print("\nPathway count per MP:")
print(mp_pathway_counts)

# 3. Combine into one summary table
summary = pd.DataFrame({
    "max_gene_score": mp_max_score,
    "n_pathways": mp_pathway_counts
}).fillna(0)

# 4. Add quality metrics if available
metrics = results_mp["metrics"]
summary = summary.join(metrics[["sampleCoverage", "silhouette", "meanSimilarity"]], how="left")

print("\nFull summary:")
print(summary.sort_values("silhouette", ascending=False))

Max gene score per MP:
mp
MP10    0.038601
MP3     0.036616
MP4     0.028645
MP12    0.026373
MP5     0.023226
MP13    0.020009
MP9     0.019947
MP30    0.019576
MP21    0.018713
MP28    0.016158
MP20    0.015923
MP7     0.015777
MP2     0.015442
MP22    0.015379
MP8     0.014775
MP16    0.014106
MP19    0.013775
MP1     0.012087
MP11    0.012065
MP24    0.009532
MP6     0.006823
MP18    0.006255
MP25    0.005437
MP14    0.005118
MP27    0.004514
MP29    0.004054
MP23    0.002509
MP15    0.002117
MP17    0.001589
MP26    0.000294
Name: score, dtype: float64

Pathway count per MP:
MP
MP26    4422
MP17    2197
MP23    2053
MP29    1631
MP6      777
MP18     753
MP11     727
MP15     667
MP2      569
MP4      471
MP14     405
MP27     379
MP1      377
MP30     325
MP20     305
MP16     284
MP8      270
MP5      253
MP19     239
MP9      227
MP10     197
MP13     186
MP21     154
MP7      146
MP28     138
MP3       56
MP12      15
MP25       1
dtype: int64

Full summary:
      max_gene_sco

In [11]:
mp_pathway_counts = gsea_df.groupby("MP").size().sort_values(ascending=False)
print("Pathway count per MP:")
print(mp_pathway_counts)

summary = pd.DataFrame({
    "max_gene_score": mp_max_score,
    "n_pathways": mp_pathway_counts
}).fillna(0)

metrics = results_mp["metrics"]
summary = summary.join(metrics[["sampleCoverage", "silhouette", "meanSimilarity"]], how="left")

print("\nFull summary:")
print(summary.sort_values("silhouette", ascending=False))

Pathway count per MP:
MP
MP26    4422
MP17    2197
MP23    2053
MP29    1631
MP6      777
MP18     753
MP11     727
MP15     667
MP2      569
MP4      471
MP14     405
MP27     379
MP1      377
MP30     325
MP20     305
MP16     284
MP8      270
MP5      253
MP19     239
MP9      227
MP10     197
MP13     186
MP21     154
MP7      146
MP28     138
MP3       56
MP12      15
MP25       1
dtype: int64

Full summary:
      max_gene_score  n_pathways  sampleCoverage  silhouette  meanSimilarity
MP11        0.012065       727.0        0.303419    0.405800           0.519
MP12        0.026373        15.0        0.264957    0.393355           0.490
MP2         0.015442       569.0        0.495726    0.357757           0.541
MP24        0.009532         0.0        0.038462    0.340832           0.361
MP3         0.036616        56.0        0.341880    0.335111           0.433
MP7         0.015777       146.0        0.081197    0.284584           0.427
MP1         0.012087       377.0        0.14

In [15]:
results_mp_loaded = kaleidocell.load("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea/results_mp.kc")
print(results_mp_loaded.keys())

dict_keys(['cluster_dict', 'mp_dict', 'mp_df', 'similarity_matrix_sorted', 'metrics'])


In [16]:

resultsk10_mp_loaded = kaleidocell.load("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_5_nmf_k10/results_mp_k10.kc")
print(resultsk10_mp_loaded.keys())

dict_keys(['cluster_dict', 'mp_dict', 'mp_df', 'similarity_matrix_sorted', 'metrics'])


In [13]:
top_gsea_per_mp = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_1_top_gsea_per_mp.csv")

# Which MPs are represented (i.e. have meaningful pathways)?
meaningful_mps = top_gsea_per_mp["MP"].unique()
print(f"MPs with meaningful pathways: {meaningful_mps}")

# Filter summary to only these
summary_meaningful = summary.loc[summary.index.isin(meaningful_mps)]
print(summary_meaningful.sort_values("silhouette", ascending=False))

MPs with meaningful pathways: ['MP1' 'MP2' 'MP3' 'MP4' 'MP5' 'MP6' 'MP7' 'MP8' 'MP9' 'MP10' 'MP11'
 'MP12' 'MP13' 'MP14' 'MP15' 'MP16' 'MP17' 'MP18' 'MP19' 'MP20' 'MP21'
 'MP23' 'MP25' 'MP26' 'MP27' 'MP28' 'MP29' 'MP30']
      max_gene_score  n_pathways  sampleCoverage  silhouette  meanSimilarity
MP11        0.012065       727.0        0.303419    0.405800           0.519
MP12        0.026373        15.0        0.264957    0.393355           0.490
MP2         0.015442       569.0        0.495726    0.357757           0.541
MP3         0.036616        56.0        0.341880    0.335111           0.433
MP7         0.015777       146.0        0.081197    0.284584           0.427
MP1         0.012087       377.0        0.149573    0.279149           0.491
MP10        0.038601       197.0        0.230769    0.230069           0.429
MP20        0.015923       305.0        0.047009    0.216024           0.337
MP4         0.028645       471.0        0.205128    0.146454           0.327
MP19     

In [13]:
summary_meaningful = summary.loc[summary.index.isin(meaningful_mps)].copy()

# Kombinierter Score: z.B. Rang-basiert über mehrere Metriken
summary_meaningful["rank_silhouette"] = summary_meaningful["silhouette"].rank(ascending=False)
summary_meaningful["rank_max_gene_score"] = summary_meaningful["max_gene_score"].rank(ascending=False)
summary_meaningful["rank_n_pathways"] = summary_meaningful["n_pathways"].rank(ascending=False)
summary_meaningful["combined_rank"] = summary_meaningful[
    ["rank_silhouette", "rank_max_gene_score", "rank_n_pathways"]
].mean(axis=1)

top_mps = summary_meaningful.sort_values("combined_rank").head(10)
print(top_mps)

      max_gene_score  n_pathways  sampleCoverage  silhouette  meanSimilarity  \
MP4         0.028645       471.0        0.205128    0.146454           0.327   
MP2         0.015442       569.0        0.495726    0.357757           0.541   
MP11        0.012065       727.0        0.303419    0.405800           0.519   
MP10        0.038601       197.0        0.230769    0.230069           0.429   
MP3         0.036616        56.0        0.341880    0.335111           0.433   
MP12        0.026373        15.0        0.264957    0.393355           0.490   
MP20        0.015923       305.0        0.047009    0.216024           0.337   
MP5         0.023226       253.0        0.273504    0.098320           0.283   
MP30        0.019576       325.0        0.260684    0.088221           0.255   
MP1         0.012087       377.0        0.149573    0.279149           0.491   

      rank_silhouette  rank_max_gene_score  rank_n_pathways  combined_rank  
MP4               9.0                  3.0